# Building Footprint Extraction
**NB 02** | Dubai, UAE | Sentinel-2 10m | SegFormer-B2

In [2]:
import warnings; warnings.filterwarnings('ignore')
import json
import pathlib

import numpy as np

import pygeovision as pgv
from pygeovision.inference.tiled import TiledInference
from pygeovision.models import get_model

client = pgv.PyGeoVision()
print(client)


14:31:45 INFO [      engine] PyGeoFetch ready


2026-07-16 14:31:45,441 [INFO] pygeofetch.core.engine: PyGeoFetch ready


14:31:45 INFO [      engine] PyGeoFetch ready


2026-07-16 14:31:45,443 [INFO] pygeofetch.core.engine: PyGeoFetch ready
2026-07-16 14:31:45,444 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v1.0 Python API active — using native search/download
2026-07-16 14:31:45,444 [INFO] pygeovision.data.pgf_bridge: PyGeoFetch v2.0 processing engine detected — using native preprocessing


PyGeoVision(v2.1.6 | pygeofetch=✓ | pgf_v2=✓v2 | geoai=independent | datasets=503 | models=98 | pipelines=51 | validator=✓ | preprocess=✓ | indices=22 | postprocess=✓ | sar=✓ | labeling=7src | losses=10 | inference=4 | xai=4 | monitoring=3 | cloud=3·aws·azure·gcp | edge=ONNX+Jetson | vlm=CLIP+Moon | few_shot | timeseries | 3D)


In [3]:
BBOX       = (55.25, 25.15, 55.35, 25.25)
DATE_RANGE = ('2024-01-01', '2024-03-31')
PROVIDERS  = ['planetary_computer']

DATA_DIR = pathlib.Path('./results/notebook')
DATA_DIR.mkdir(parents=True, exist_ok=True)

results = client.search(
    bbox            = BBOX,
    date_range      = DATE_RANGE,
    providers       = PROVIDERS,
    cloud_cover_max = 10,
)
print(f"Found {len(results)} scenes")
for r in results[:5]:
    print(f"  {r.provider:<22} {r.datetime[:10]}  cloud={r.cloud_cover:.1f}%  {r.id[:40]}")


Found 32 scenes
  planetary_computer     2024-03-18  cloud=0.1%  LC09_L2SP_160043_20240318_02_T1
  planetary_computer     2024-03-18  cloud=0.1%  S2A_MSIL2A_20240318T064621_R020_T40RCN_2
  planetary_computer     2024-03-18  cloud=8.6%  S2A_MSIL2A_20240318T064621_R020_T40RCP_2
  planetary_computer     2024-03-13  cloud=0.1%  S2B_MSIL2A_20240313T064639_R020_T40RCP_2
  planetary_computer     2024-03-13  cloud=0.2%  S2B_MSIL2A_20240313T064639_R020_T40RCN_2


In [ ]:
BANDS = ['B02', 'B03', 'B04', 'B08']

downloads = client.download(
    results[:1],
    output_dir   = str(DATA_DIR),
    bands        = BANDS,
    post_process = ['reproject:EPSG:32640', 'cog'],
)
scene_path = downloads[0].path if downloads and downloads[0].success else None
scl_cands  = list(DATA_DIR.rglob('*SCL*.tif')) + list(DATA_DIR.rglob('*scl*.tif'))
scl_path   = str(scl_cands[0]) if scl_cands else None
if scene_path:
    d = downloads[0]
    print(f"Downloaded   : {scene_path}")
    print(f"Size         : {d.bytes_downloaded/1024/1024:.1f} MB")
    print(f"SCL mask     : {scl_path}")
else:
    print("Download failed or scene unavailable — check provider availability")



  📡 Downloading 1 scene(s) via PyGeoFetch from planetary_computer
  📁 Output: results/notebook
  🔧 Post-process: reproject:EPSG:32640 → cog
  ⚡ Parallel downloads: 4

  ⠋  Downloading 1 scene(s)…  
  ┌──────────────────────────────────────────────────────────────────────┐
  │  ⬇  DOWNLOADING  1 scene  → results/notebook                         │
  └──────────────────────────────────────────────────────────────────────┘

14:18:04 INFO [  downloader] Downloading LC09_L2SP_160043_20240318_02_T1


2026-07-16 14:18:04,361 [INFO] pygeofetch.core.downloader: Downloading LC09_L2SP_160043_20240318_02_T1


  ⠇  Downloading 1 scene(s)…  14:18:05 INFO [planetary_computer] Downloading 19 asset(s) for LC09_L2SP_160043_20240318_02_T1: ['qa', 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'qa_pixel', 'qa_radsat', 'qa_aerosol']


2026-07-16 14:18:05,378 [INFO] pygeofetch.provider.planetary_computer: Downloading 19 asset(s) for LC09_L2SP_160043_20240318_02_T1: ['qa', 'red', 'blue', 'drad', 'emis', 'emsd', 'trad', 'urad', 'atran', 'cdist', 'green', 'nir08', 'lwir11', 'swir16', 'swir22', 'coastal', 'qa_pixel', 'qa_radsat', 'qa_aerosol']


14:18:05 INFO [planetary_computer]   Fetching asset 'qa' → LC09_L2SP_160043_20240318_20240319_02_T1_ST_QA.TIF


2026-07-16 14:18:05,379 [INFO] pygeofetch.provider.planetary_computer:   Fetching asset 'qa' → LC09_L2SP_160043_20240318_20240319_02_T1_ST_QA.TIF


  ⠧  Downloading 1 scene(s)…  

In [3]:
if scene_path:
    raw_report = client.validator.validate(
        str(scene_path),
        required_bands = 4,
        value_range    = (0, 65535),
    )
    print("Raw data validation:")
    print(raw_report.summary())
    if not raw_report.passed:
        print("WARNING: Issues found — preprocessing will auto-fix")


NameError: name 'scene_path' is not defined

In [ ]:
PREPROCESSED = DATA_DIR / 'dubai_preprocessed.tif'

if scene_path:
    ready = client.prepare_for_ai(
        str(scene_path),
        stack_bands  = BANDS,
        bbox         = BBOX,
        scl_path         = scl_path,
        scl_keep_classes = [4,5,6],
        normalise    = 'scale_factor',
        scale_factor = 10000.0,
        model_type   = 'segmentation',
        output_path  = str(PREPROCESSED),
    )
    print("Preprocessing steps :", ready['steps'])
    print("Output shape (C,H,W):", ready['shape'])
    print("Resolution          :", ready['resolution_m'], "m")
    print("Validation          :", "PASSED" if ready['report'] and ready['report'].passed else "FIXED (auto)")
    arr = ready['array']
    print(f"Value range         : {arr.min():.4f} → {arr.max():.4f}")
    print(f"NaN count           : {np.isnan(arr).sum()}")
else:
    print("No scene available — cannot preprocess")
    ready = None


In [5]:
# Auto-labels from Microsoft Buildings + OSM
LABELS_DIR = DATA_DIR / 'labels'
LABELS_DIR.mkdir(exist_ok=True)
import pygeovision.labeling.osm as osm_module

# Patch the default URL
osm_module.OVERPASS_URL = "https://overpass.openstreetmap.fr/api/interpreter"
# Or set the instance's attribute after creation
labeler = osm_module.OSMLabeler()
labeler.overpass_url = "https://overpass.openstreetmap.fr/api/interpreter"
# Then call label
ms = labeler.label(bbox=BBOX, categories=['buildings'], output_path=str(LABELS_DIR/'ms_buildings.tif'))
q  = client.labeling.quality(str(LABELS_DIR/'ms_buildings.tif'))
print(f"Label quality: {q['quality_grade']}  score={q['quality_score']:.2f}")


2026-07-16 14:32:05,477 [INFO] pygeovision.labeling.osm: OSMLabeler: querying Overpass for bbox=(55.25, 25.15, 55.35, 25.25) cats=['buildings']
2026-07-16 14:32:06,437 [WARNING] pygeovision.labeling.osm: Overpass https://overpass-api.de/api/interpreter HTTP 406 (attempt 1)
2026-07-16 14:32:09,613 [WARNING] pygeovision.labeling.osm: Overpass https://overpass.kumi.systems/api/interpreter HTTP 429 (attempt 1)
2026-07-16 14:32:11,660 [WARNING] pygeovision.labeling.osm: Overpass https://overpass.openstreetmap.fr/api/interpreter HTTP 403 (attempt 1)
2026-07-16 14:32:16,249 [WARNING] pygeovision.labeling.osm: Overpass https://overpass-api.de/api/interpreter HTTP 406 (attempt 2)
2026-07-16 14:32:22,313 [WARNING] pygeovision.labeling.osm: Overpass https://overpass.kumi.systems/api/interpreter HTTP 429 (attempt 2)
2026-07-16 14:32:29,380 [WARNING] pygeovision.labeling.osm: Overpass https://overpass.openstreetmap.fr/api/interpreter HTTP 403 (attempt 2)
2026-07-16 14:32:36,855 [WARNING] pygeovisio

TypeError: unsupported format string passed to NoneType.__format__

In [ ]:
import geopandas as gpd
import numpy as np


def download_ms_buildings_bbox(bbox, output_path, zoom=9):
    """Download Microsoft building footprints for a bounding box."""
    import gzip
    import json
    import urllib.request

    import mercantile

    min_lon, min_lat, max_lon, max_lat = bbox

    # Get quadkeys covering the bbox
    tiles = list(mercantile.tiles(min_lon, min_lat, max_lon, max_lat, zooms=zoom))

    all_features = []
    for tile in tiles:
        quadkey = mercantile.quadkey(tile)
        url = f"https://minedbuildings.blob.core.windows.net/global-buildings/{quadkey}.geojsonl.gz"

        try:
            # Download the file
            urllib.request.urlretrieve(url, f"/tmp/{quadkey}.geojsonl.gz")

            # Parse the GeoJSONL file
            with gzip.open(f"/tmp/{quadkey}.geojsonl.gz", "rt") as f:
                for line in f:
                    if line.strip():
                        feature = json.loads(line)
                        all_features.append(feature)
            print(f"Loaded {len(all_features)} buildings from quadkey {quadkey}")
        except Exception as e:
            print(f"Failed to download {quadkey}: {e}")

    # Convert to GeoDataFrame
    import shapely.geometry as sg
    geometries = []
    for feature in all_features:
        try:
            geom = sg.shape(feature["geometry"])
            geometries.append(geom)
        except:
            continue

    gdf = gpd.GeoDataFrame(geometry=geometries, crs="EPSG:4326")

    # Clip to bbox
    bbox_poly = sg.box(*bbox)
    gdf = gdf[gdf.intersects(bbox_poly)]

    print(f"Total buildings in bbox: {len(gdf)}")
    return gdf

# Usage
gdf = download_ms_buildings_bbox(BBOX, LABELS_DIR / 'buildings.geojson')

NameError: name 'LABELS_DIR' is not defined

In [ ]:
PREDICTION = DATA_DIR / 'buildings_raw.tif'

if ready and PREPROCESSED.exists():
    model = get_model('segformer-b2', num_classes=2, pretrained=True)
    inf   = TiledInference(model, chip_size=512, overlap=64, blend_mode='gaussian')
    res   = inf.infer(str(PREPROCESSED), str(PREDICTION))
    print(f"Inference: {res.get('n_chips')} chips  {res.get('duration_seconds',0):.1f}s  "
          f"{res.get('chips_per_second',0):.1f} chips/s")
else:
    print("Skipping inference — no preprocessed scene")


In [ ]:
if PREDICTION.exists():
    pred_report = client.validator.validate(str(PREDICTION))
    print("Prediction validation:", "PASSED" if pred_report.passed else f"FIXED — {pred_report.errors}")
    print(pred_report.summary())


In [ ]:
SIEVED  = DATA_DIR / 'sieved.tif'
VECTOR  = DATA_DIR / 'output.geojson'
COG_OUT = DATA_DIR / 'output_cog.tif'

if PREDICTION.exists():
    client.postprocess.sieve_filter(str(PREDICTION), min_pixels=25, output_path=str(SIEVED))
    client.postprocess.vectorise(str(SIEVED), str(VECTOR),
        target_class=1, min_area_m2=50.0, simplify_tolerance=0.3)
    client.postprocess.regularise_buildings(str(VECTOR), str(DATA_DIR/'regularised.geojson'))
    stats = client.postprocess.class_statistics(str(SIEVED))
    client.postprocess.to_cog(str(SIEVED), str(COG_OUT))
    with open(VECTOR) as f: gj = json.load(f)
    print(f"Polygons detected : {len(gj['features'])}")
    for cls, info in sorted(stats.items()):
        print(f"  Class {cls}: {info['pixels']:>8,} px | {info['area_ha']:>8.1f} ha | {info['pct']:>6.1f}%")
    print(f"COG saved: {COG_OUT}")


**NB02 — Building Extraction** complete.
- Study area: Dubai UAE
- Sensor: Sentinel-2 10m
- Model: SegFormer-B2
- Pipeline: search → download → validate_raw → preprocess → validate_clean → model → postprocess → COG